# NB4 · Q-learning 复现：seed、敏感性与误差带

对应主站 **L7（时序差分方法）**，难度 ★★☆（level 2），预计 **50 分钟**。这是本站"科研训练"的重点本之一——前半本你把算法跑对，后半本你像做实验一样对待它。

**前置**：主站 L7 各小节，尤其 qa 节的 **Dvoretzky 定理推导**（Q-learning 收敛证明的靠山——目标会动的一般 RM 定理）；NB0（环境复刻）或对 4×4 作业世界熟悉。

一句话立场，也是本笔记本的副标题：

> **单次训练曲线没有意义，方差才是第一公民。**

你将亲手把这句话验证四遍：

1. **TODO 1** —— 写出 seed 可调的 `train()`，复现 Q-learning；
2. **TODO 2** —— 步长 α 敏感性（α ∈ {0.1, 0.5, 0.9}，每档 5 seed）；
3. **TODO 3** —— 探索率 ε 敏感性：训练期探索 vs 评估期 greedy 的解耦；
4. **TODO 4** —— 10-seed 均值 ± 1.96·std/√10 的**误差带**——论文图里那块阴影的来历。

最后留一个无答案的 🏔 挑战：ε 线性衰减 vs 恒定，谁的样本效率高？用多 seed 说服自己。

## 怎么用这本笔记本

- 带 **TODO** 的格子是留给你的：按提示补完代码（签名、关键行都给了），然后删掉 `raise NotImplementedError`；
- 带 **✅ 自检** 的格子是判分器：assert 通过并打出 ✅，这一节才算毕业。**所有阈值都对着"理论最优 + 实测裕度"校准过**；
- 🏔 挑战格无答案、无 assert——那是留给你的科研题；
- 所有随机性都走 `np.random.default_rng(seed)`：**同一个 seed，永远同一条曲线**——这是"复现"两个字的全部含义；
- 已知限制：浏览器刷新清空 kernel 状态，回来后 Run All 重跑；本本要画图，首启需多下载 matplotlib（约 8–10MB，之后有缓存）；
- 图内文字用英文——浏览器内核的 matplotlib 没有中文字体，中文标签会变成豆腐块。

## 0 · 准备

numpy 负责计算，matplotlib 负责把"方差"画出来给你看。

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("numpy      :", np.__version__)
print("matplotlib :", matplotlib.__version__)
assert int(np.__version__.split(".")[0]) in (1, 2), "numpy 主版本异常"
assert int(matplotlib.__version__.split(".")[0]) == 3, "matplotlib 主版本异常"
print("✅ 依赖就绪（numpy + matplotlib）")

## 1 · 世界：4×4 作业世界（NB0 同款）

规格与主站 L1「代码精讲」和 NB0 完全一致：

| 项目 | 值 |
|---|---|
| 网格 | 4×4，状态 s1–s16，起点 **s1** |
| 禁区 | **s8、s10** —— 撞上：**原地弹回**，奖励 −1（作业代码规则，非"可进入只扣分"） |
| 目标 | **s12** —— 进入：+1，回合结束 |
| 出界 | 原地不动，奖励 −1 |
| 折扣 | γ = 0.9 |

动作 5 个（作业代码列序）：**下、右、上、左、原**。

先算一笔账，后面所有阈值都靠它。从 s1 到 s12 的最短路径是 **5 步**——例如 右右下下右（s1→s2→s3→s7→s11→s12）；s8、s10 两个禁区把所有 4 步路线都堵死了。所以本世界的**理论最优回报**

$$G^* = \gamma^4 = 0.9^4 = 0.6561$$

任何策略的折扣回报都不会超过它。记住这个数。

In [ ]:
SIZE = 4
NUM_STATES = SIZE * SIZE                 # s1 .. s16
START, TARGET = 1, 12
FORBIDDEN = {8, 10}
REWARDS = {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}

GAMMA = 0.9
EPISODES = 500                           # 默认训练回合数（自检 1 的口径）
MAX_STEPS = 200                          # 单回合步数上限（防不收敛回合拖死循环）

ACTION_SPACE = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]   # 下 右 上 左 原
DOWN, RIGHT, UP, LEFT, STAY = ACTION_SPACE

def s2xy(s):
    """状态编号 → (x, y)，y 向下增长：s1=(0,0) s8=(3,1) s10=(1,2) s12=(3,2)"""
    i = int(s) - 1
    return i % SIZE, i // SIZE

def xy2s(x, y):
    """(x, y) → 状态编号"""
    return int(y) * SIZE + int(x) + 1

assert (SIZE, NUM_STATES, START, TARGET) == (4, 16, 1, 12)
assert FORBIDDEN == {8, 10} and GAMMA == 0.9
assert s2xy(8) == (3, 1) and s2xy(10) == (1, 2) and s2xy(12) == (3, 2)
assert all(xy2s(*s2xy(s)) == s for s in range(1, 17))
print("✅ 世界规格就绪：4×4 / 起点 s1 / 禁区 s8,s10（弹回）/ 目标 s12 / γ=0.9")

In [ ]:
class GridWorld:
    """4×4 网格世界：NB0 同款（书配 grid_world.py 的 numpy 瘦身复刻）。

    分支优先级：出界 > 目标 > 禁区 > 普通（if/elif 短路，每个动作只命中第一个匹配分支）：
      出界    → 原地不动，reward = boundary  = -1
      进目标  → 走进目标，reward = target   = +1，done=True
      撞禁区  → 原地弹回，reward = forbidden = -1
      普通/原 → 正常移动，reward = other    =  0
    """

    def __init__(self, size=SIZE, start=START, target=TARGET, forbidden=FORBIDDEN):
        self.size = size
        self.num_states = size * size
        self.start_state, self.target_state = start, target
        self.forbidden_states = set(forbidden)
        self.action_space = ACTION_SPACE
        self.agent_state = start

    def reset(self):
        self.agent_state = self.start_state
        return self.agent_state

    def _get_next_state_and_reward(self, state, action):
        x, y = s2xy(state)
        nxt = np.array([x, y]) + np.array(action)
        if not (0 <= nxt[0] < self.size and 0 <= nxt[1] < self.size):
            next_state, reward = state, REWARDS["boundary"]        # 1) 出界：原地
        elif xy2s(nxt[0], nxt[1]) == self.target_state:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["target"]    # 2) 目标
        elif xy2s(nxt[0], nxt[1]) in self.forbidden_states:
            next_state, reward = state, REWARDS["forbidden"]       # 3) 禁区：弹回
        else:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["other"]     # 4) 普通
        return next_state, reward

    def step(self, action):
        assert any(tuple(action) == a for a in self.action_space), "非法动作"
        next_state, reward = self._get_next_state_and_reward(self.agent_state, action)
        done = next_state == self.target_state
        self.agent_state = next_state
        return next_state, reward, done, {}

env = GridWorld()
assert env.reset() == START and env.num_states == 16 and len(env.action_space) == 5
print("✅ 环境就绪：reset() → s1")

In [ ]:
# 三条锚点转移，对应主站 L1 代码精讲的作业代码规则
env.reset()
ns, r, done, _ = env.step(UP)                 # s1 向上：出界
print(f"s1  --up-->    s{ns}  reward={r:+.0f}  done={done}")
assert (ns, r, done) == (1, -1.0, False), "出界：−1 且留在 s1"

env.agent_state = 11
ns, r, done, _ = env.step(RIGHT)              # s11 右移进 s12：目标
print(f"s11 --right--> s{ns}  reward={r:+.0f}  done={done}")
assert (ns, r, done) == (12, 1.0, True), "进目标：+1 且 done"

env.agent_state = 7
ns, r, done, _ = env.step(RIGHT)              # s7 右移撞 s8：禁区弹回
print(f"s7  --right--> s{ns}  reward={r:+.0f}  done={done}")
assert (ns, r, done) == (7, -1.0, False), "撞禁区：−1 且弹回原地（作业代码规则）"
print("✅ 环境语义对齐：出界 −1 / 目标 +1 终止 / 禁区弹回 −1")

## 2 · 两个策略：行为 ε-greedy，评估 greedy

L7 的核心解耦，在本笔记本里长这样：

- **行为策略**（训练期走路用）：ε-greedy——以 ε 概率均匀随机，否则贪心 argmax。它只负责一件事：**把每个 (s, a) 都送到**（收敛条件①充分探索）。
- **评估策略**（测学得好不好用）：纯 greedy——无探索、确定性，从 s1 出发走到头，算折扣回报。它才是 Q-learning 真正估计的对象：最优动作价值 q*（收敛条件②步长条件，靠 Dvoretzky 定理兜底——见主站 L7 qa 节推导）。

两者分离正是 **off-policy** 的含义：行为策略再烂，不妨碍学出来的 greedy 策略最优——第 6 节你会把这句话测到 ε=1.0 的极端。

两个函数本笔记本直接给出（不用你写），但每一条 assert 都值得读一遍。

In [ ]:
def eps_greedy_action(rng, q_row, epsilon):
    """ε-greedy 行为策略：以 ε 概率均匀随机探索；否则贪心 argmax（平局随机打破——与主站 L7 实验室同款）。"""
    if rng.random() < epsilon:
        return int(rng.integers(len(q_row)))
    best = np.flatnonzero(q_row == q_row.max())
    return int(best[rng.integers(len(best))])

# ε=0 时纯贪心：两个并列最大值（0.9）之间随机打破
rng_t = np.random.default_rng(0)
picks = {eps_greedy_action(rng_t, np.array([0.1, 0.9, 0.3, 0.9, 0.0]), 0.0) for _ in range(50)}
assert picks == {1, 3}, "平局应只在两个 argmax 之间随机"

# ε=1 时纯随机：动作下标永远合法
assert all(0 <= eps_greedy_action(np.random.default_rng(7), np.zeros(5), 1.0) < 5 for _ in range(50))

# 同 seed 必同序列（复现性的最小验证）
seq1 = [eps_greedy_action(np.random.default_rng(3), np.zeros(5), 1.0) for _ in range(10)]
seq2 = [eps_greedy_action(np.random.default_rng(3), np.zeros(5), 1.0) for _ in range(10)]
assert seq1 == seq2
print("✅ ε-greedy 采样就绪（随机平局 + 全程 rng 种子化）")

In [ ]:
def greedy_return(Q, from_state=START, gamma=GAMMA, max_steps=200):
    """评估：从 from_state 出发全程 argmax(Q)（无探索），返回折扣回报。到不了目标就返回累计值（必为负）。"""
    env.agent_state = from_state
    G, disc = 0.0, 1.0
    for _ in range(max_steps):
        a = int(np.argmax(Q[env.agent_state - 1]))
        _, r, done, _ = env.step(ACTION_SPACE[a])
        G += disc * r
        disc *= gamma
        if done:
            return G
    return G

# 手工摆一张"最优路线"Q 表验证：s1→s2→s3→s7→s11→s12（右右下下右，5 步）
Q_hand = np.zeros((NUM_STATES, 5))
for s, a in [(START, RIGHT), (2, RIGHT), (3, DOWN), (7, DOWN), (11, RIGHT)]:
    Q_hand[s - 1, ACTION_SPACE.index(a)] = 1.0
assert abs(greedy_return(Q_hand) - GAMMA ** 4) < 1e-12, "greedy 评估应算出 5 步最优路线的折扣回报"
print(f"✅ greedy 评估就绪：手工最优路线 5 步，G* = {GAMMA ** 4:.4f}（本世界理论最优）")

## 3 · 训练函数（TODO 1）

Q-learning 的全部灵魂是一行更新：

$$Q(s,a) \;\leftarrow\; Q(s,a) + \alpha\,\Big(\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{TD 目标（done 时不含 bootstrap 项）}} - Q(s,a)\Big)$$

动手前，三个实现决策（坑都替你踩过了）：

1. **乐观初始化 Q₀ = 1.0**，而不是全零。为什么：全零 + `np.argmax` 的平局规则会永远偏向下标 0 的动作（"下"），智能体开局一头往南墙撞，ε=0.1 的探索在 500 回合内救不回来（实测长期卡在 0.53 的次优路线）。乐观初始化让"没试过的动作看起来更香"，探索从碰运气变成系统性拆迁（Sutton & Barto §2.6 的经典技巧）。
2. **argmax 平局随机打破**——`eps_greedy_action` 已经替你实现。
3. **全部随机性只走 rng(seed)**：函数内不碰任何全局随机源，seed 定了曲线就定了。

另一个高频坑：到达目标（done）时 TD 目标就是纯 r——**不要** bootstrap 站在目标格上的 Q（终态价值定义为 0）。

In [ ]:
def train(seed=42, alpha=0.1, epsilon=0.1, episodes=500, gamma=GAMMA):
    """TODO 1 · Q-learning 训练。

    返回 (Q, returns)：
      Q        —— 形状 (16, 5) 的动作价值表
      returns  —— 长度 episodes 的数组，returns[e] = 第 e 个回合的折扣回报
    """
    rng = np.random.default_rng(seed)     # 本函数全部随机性只走这一个 rng

    # TODO 1a：初始化 Q 为 (16, 5) 的全 1.0 矩阵（乐观初始化，原因见上一格）

    # TODO 1b：逐回合循环（共 episodes 个回合）：
    #     s = env.reset()
    #     逐步走（回合内最多 MAX_STEPS 步）：
    #         a = eps_greedy_action(rng, Q[s-1], epsilon)                  # 行为策略
    #         ns, r, done, _ = env.step(ACTION_SPACE[a])
    #         td_target = r + (0.0 if done else gamma * np.max(Q[ns-1]))   # done 时不 bootstrap！
    #         Q[s-1, a] += alpha * (td_target - Q[s-1, a])                 # ← 主角一行
    #         G += disc * r ;  disc *= gamma ;  s = ns ;  done 则跳出
    #     returns[e] = G

    raise NotImplementedError("TODO 1：完成 train() 后删除本行")

**✅ 自检 1**：默认参数 α=0.1、ε=0.1、500 回合、seed=42 训练后，greedy 策略从 s1 出发的回合回报应 ≥ **0.6**。

阈值怎么来的：理论最优 G\* = 0.9⁴ = 0.6561；实测默认参数稳定收敛到 0.6561，阈值向下留裕度——差半步到最优也放行，学偏了则拦下。

In [ ]:
Q42, ret42 = train(seed=42, alpha=0.1, epsilon=0.1, episodes=500)
g42 = greedy_return(Q42)

print(f"greedy 策略从 s1 出发的回合回报 = {g42:.4f}   （理论最优 = 0.9^4 = {GAMMA ** 4:.4f}）")
print(f"训练期最后 50 回合平均回报       = {ret42[-50:].mean():.4f}   （含 ε=0.1 探索损耗，低于 greedy 评估）")
assert g42 >= 0.6, "默认参数下 greedy 评估应 >= 0.6"
print("✅ 自检 1 通过：默认参数（α=0.1, ε=0.1, 500 回合, seed=42）学到了（近）最优策略")

## 4 · 单 seed 曲线：它很抖

先看一条曲线：raw 回报逐回合画出来（浅灰），叠一条窗口 20 的滑动平均（蓝）。注意两件事：

- **收敛之后它依然抖**：ε=0.1 的探索每回合都可能撞墙挨 −1，单回合回报永远是个随机变量；
- **它只是一个 seed 的故事**：换一个 seed，开局与爬坡的形状都会变。从下一节起，我们不再看单条线。

In [ ]:
SMOOTH_W = 20   # 滑动平均窗口

def smooth(x, w=SMOOTH_W):
    """滑动平均：返回长度 len(x)-w+1 的曲线（第 k 个点 = 第 k..k+w-1 回合的均值）。"""
    return np.convolve(x, np.ones(w) / w, mode="valid")

probe = smooth(np.arange(30, dtype=float))
assert len(probe) == 30 - SMOOTH_W + 1 and abs(probe[0] - np.arange(SMOOTH_W).mean()) < 1e-12

raw = ret42
cur = smooth(raw)
xs = np.arange(len(cur)) + SMOOTH_W - 1

fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
ax.plot(raw, color="lightgray", lw=0.8, label="raw episode return")
ax.plot(xs, cur, color="#4c72b0", lw=2, label=f"smoothed (window {SMOOTH_W})")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set_xlabel("episode")
ax.set_ylabel("return")
ax.set_title("One seed is one story: seed=42, alpha=0.1, eps=0.1")
ax.legend()
plt.show()

print(f"最后 100 回合 raw 回报的标准差 = {raw[-100:].std():.3f} —— 收敛了，但每回合仍在上下跳")
print("而这条曲线，只是 seed=42 一个 seed 的故事。")

## 5 · α 敏感性（TODO 2）

步长 α 是 RM 骨架里最敏感的旋钮（L6 的步长三命运）。教科书图景是"大步长学得快但抖、小步长稳但慢"。但本世界有个特殊性必须先说清：

**这个环境是确定性的**——给定 (s, a)，r 与 s′ 没有任何噪声；TD 目标里唯一的随机性来自探索带来的访问顺序。经典图景中"大 α 早期剧烈抖动"的那一面（它需要随机奖励来激发）在这里退场，你会看到的是它的另一面：

- **早期**：α 越大，价值沿路线传得越快，回报爬坡越早（样本效率）；
- **稳态**：α 越大，Q 表在收敛点附近的小幅震荡越明显，回报的**后期**波动更大（主站 L7 结尾"小常数步长的波动是可以接受的代价"的反面——大常数步长的波动，得自己扛）。

自检 2 的三条 assert 就按这个真实图景校准。

In [ ]:
ALPHAS = [0.1, 0.5, 0.9]          # 步长扫描档位
SWEEP_SEEDS = [1, 2, 3, 4, 5]     # 每档 5 个 seed

# TODO 2：对每个 α × 每个 seed 调 train()（ε 固定 0.1，episodes=EPISODES），
# 把每回合回报序列收进 alpha_returns —— 形状 (len(ALPHAS), len(SWEEP_SEEDS), EPISODES)

alpha_returns = None   # TODO 2：改成 np.zeros((len(ALPHAS), len(SWEEP_SEEDS), EPISODES)) 并填入双重循环

raise NotImplementedError("TODO 2：完成 α 敏感性扫描后删除本行")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
for i, alpha_ in enumerate(ALPHAS):
    cur = smooth(alpha_returns[i].mean(axis=0))       # 对 5 个 seed 取均值再平滑
    xs = np.arange(len(cur)) + SMOOTH_W - 1
    ax.plot(xs, cur, label=f"alpha={alpha_}")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set_xlabel("episode")
ax.set_ylabel("smoothed return (mean of 5 seeds)")
ax.set_title(f"alpha sweep, eps=0.1 (smoothing window {SMOOTH_W})")
ax.legend()
plt.show()

print("看两点：早期爬坡速度随 α 单调变快；末端三档都稳住了（差异在细节里，下一格量化）。")

**✅ 自检 2**（三条，全部对着实测校准）：

1. 三档 α 的**末端回报**（后 50 回合均值）都 ≥ **0.35**——哪个步长都没学崩；
2. **早期样本效率**：α=0.9 的前 50 回合均值 **>** α=0.1 的——大步长学得快；
3. **稳态波动**：α=0.9 的后 100 回合回报标准差 **>** α=0.1 的——大步长收敛后更抖。

（教材里"大 α 早期更抖"的图景需要随机奖励激发；在确定性环境里它表现为稳态抖。若你给环境加上随机奖励重做本节，把第 3 条的窗口换回早期即可看到经典图景。）

In [ ]:
end50 = alpha_returns[:, :, -50:].mean(axis=(1, 2))      # 末端回报：每档 5 seed × 后 50 回合的总平均
first50 = alpha_returns[:, :, :50].mean(axis=(1, 2))     # 早期水平：前 50 回合总平均
late_std = alpha_returns[:, :, -100:].std(axis=2).mean(axis=1)   # 稳态波动：每 seed 后 100 回合 std，再对 seed 平均

for i, alpha_ in enumerate(ALPHAS):
    print(f"alpha={alpha_}:  末端(后50均值)={end50[i]:+.4f}   前50均值={first50[i]:+.4f}   稳态std(后100)={late_std[i]:.4f}")

assert (end50 >= 0.35).all(), "三档末端回报都应 >= 0.35"
assert first50[2] > first50[0], "早期样本效率：alpha=0.9 的前 50 回合均值应高于 alpha=0.1"
assert late_std[2] > late_std[0], "稳态波动：alpha=0.9 的后期回报波动应大于 alpha=0.1"
print("✅ 自检 2 通过：三档都收敛；大步长学得快，但稳态更抖")

## 6 · ε 敏感性（TODO 3）：训练期探索 vs 评估期 greedy

ε 拧到多大算好？先分清两个对象，否则全是鸡同鸭讲：

- **训练期回报**（行为策略自己走出来的）：ε 越大越惨——ε=1.0 就是均匀随机游走，NB0 测过它从 s1 出发的价值 ≈ −2.99；
- **greedy 评估回报**（训练完拿 argmax 策略测的）：这才是 Q-learning 学到的东西。

off-policy 的极端测试：**ε=1.0（完全随机行为策略）训出来的 Q，greedy 评估照样应恢复到最优附近**——书上 Figure 7.4 演的就是这一幕。行为策略只需要保证每个 (s, a) 被充分送到；目标 r + γ max Q(s′,·) 里根本没有它的影子。

In [ ]:
EPSILONS = [0.1, 0.5, 1.0]        # 探索率扫描档位（1.0 = 完全随机行为策略）

# TODO 3：对每个 ε × 每个 seed（SWEEP_SEEDS，α=0.1，episodes=EPISODES）训练并评估：
#   eps_returns[i, j] = 每回合回报序列        —— 形状 (len(EPSILONS), len(SWEEP_SEEDS), EPISODES)
#   eps_eval[i, j]    = 训练完后 greedy_return(Q)   —— 评估期无探索，形状 (len(EPSILONS), len(SWEEP_SEEDS))

eps_returns = None   # TODO 3：同 TODO 2，另加 greedy 评估
eps_eval = None

raise NotImplementedError("TODO 3：完成 ε 敏感性扫描后删除本行")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

for i, eps_ in enumerate(EPSILONS):
    cur = smooth(eps_returns[i].mean(axis=0))
    xs = np.arange(len(cur)) + SMOOTH_W - 1
    ax1.plot(xs, cur, label=f"eps={eps_}")
ax1.axhline(0, color="gray", lw=0.8, ls=":")
ax1.set_xlabel("episode")
ax1.set_ylabel("smoothed return")
ax1.set_title("training time: behavior policy return")
ax1.legend()

centers = np.arange(len(EPSILONS))
ax2.bar(centers, eps_eval.mean(axis=1), yerr=eps_eval.std(axis=1), capsize=4,
        color=["#4c72b0", "#dd8452", "#55a868"], width=0.55)
ax2.axhline(GAMMA ** 4, color="gray", ls="--", lw=1, label="optimal 0.9^4")
ax2.set_xticks(centers)
ax2.set_xticklabels([f"eps={e}" for e in EPSILONS])
ax2.set_ylabel("greedy return from s1")
ax2.set_title("evaluation time: greedy policy")
ax2.legend()
plt.show()

print("左图：训练期 ε 越大越惨（ε=1.0 常年在 -3 附近）；右图：三档 greedy 评估全部贴着最优 —— 解耦。")

**✅ 自检 3**：每一档 ε、每一个 seed 的 greedy 评估回报都 ≥ **0.6**；且 ε=1.0 档满足"训练期末端回报 < 0 < greedy 评估"——训练曲线惨不忍睹与学出最优策略同时成立，这就是 off-policy。

In [ ]:
eval_mean = eps_eval.mean(axis=1)
for i, eps_ in enumerate(EPSILONS):
    print(f"eps={eps_}:  训练期末端回报(后50均值)={eps_returns[i, :, -50:].mean():+.4f}   greedy 评估均值={eval_mean[i]:.4f}")

assert (eps_eval >= 0.6).all(), "每一档、每一个 seed 的 greedy 评估回报都应 >= 0.6"
assert eps_returns[2, :, -50:].mean() < 0 < eval_mean[2], "eps=1.0：训练期为负，greedy 评估照样恢复"
print("✅ 自检 3 通过：训练期探索得再烂（ε=1.0 也在内），greedy 评估照样最优——行为/目标策略解耦")

## 7 · 10-seed 误差带（TODO 4）—— 本笔记本的主角

单条曲线是样本量为 1 的故事。报告结果的正确姿势，是在每个回合 e 处画：

$$\bar{G}_e \pm 1.96 \cdot \frac{\hat\sigma_e}{\sqrt{10}}$$

- $\bar{G}_e$：第 e 个回合处，10 个 seed 回报的均值；
- $\hat\sigma_e$：同一位置 10 个 seed 的样本标准差；
- 为什么除以 $\sqrt{10}$：均值的标准误是单个样本标准差的 $1/\sqrt{n}$（NB3 见过的方差塌缩，$\mathrm{Var}(\bar{X}) = \sigma^2/n$）；乘 1.96 是正态近似下的 95% 置信半宽。

这就是论文图里那块阴影的来历。画出它，再让 assert 替你确认两件事：数据形状对；**误差带末端不覆盖 0**——收敛后的回报显著为正，置信区间与 0 划清界限。

In [ ]:
BAND_SEEDS = np.arange(10)        # 10 个 seed：0..9

# TODO 4：默认参数（α=0.1, ε=0.1, episodes=EPISODES）下跑 10 个 seed，
# 把每回合回报序列收进 returns10 —— 形状 (10, EPISODES)

returns10 = None   # TODO 4：np.zeros((len(BAND_SEEDS), EPISODES)) + 循环

raise NotImplementedError("TODO 4：完成 10-seed 复现后删除本行")

In [ ]:
sm = np.stack([smooth(returns10[j]) for j in range(len(BAND_SEEDS))])   # (10, EPISODES-w+1)
mean_curve = sm.mean(axis=0)
band_half = 1.96 * sm.std(axis=0, ddof=1) / np.sqrt(len(BAND_SEEDS))
xs = np.arange(len(mean_curve)) + SMOOTH_W - 1

fig, ax = plt.subplots(figsize=(7.5, 4.4), constrained_layout=True)
ax.fill_between(xs, mean_curve - band_half, mean_curve + band_half,
                alpha=0.25, color="#4c72b0", label="95% CI: mean +/- 1.96*std/sqrt(10)")
ax.plot(xs, mean_curve, color="#4c72b0", lw=2, label="mean over 10 seeds")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.axhline(GAMMA ** 4, color="#55a868", ls="--", lw=1, label="optimal 0.9^4 = 0.656")
ax.set_xlabel("episode")
ax.set_ylabel("smoothed return")
ax.set_title("10-seed error band (alpha=0.1, eps=0.1)")
ax.legend(loc="lower right", fontsize=8)
plt.show()

print("注意均值曲线顶端与 0.656 之间的缺口：那是 ε=0.1 的探索损耗，不是没学到位。")

**✅ 自检 4**：`returns10` 形状恰为 **(10, 500)**；滑动平均曲线上最后一个点处，误差带下界 **> 0**（阴影末端不覆盖 0）。

In [ ]:
sm = np.stack([smooth(returns10[j]) for j in range(len(BAND_SEEDS))])
mean_curve = sm.mean(axis=0)
band_half = 1.96 * sm.std(axis=0, ddof=1) / np.sqrt(len(BAND_SEEDS))
band_lower, band_upper = mean_curve - band_half, mean_curve + band_half

assert returns10.shape == (len(BAND_SEEDS), EPISODES), "形状应为 (10, EPISODES)"
assert band_lower[-1] > 0, "误差带末端不应覆盖 0"
print(f"✅ 自检 4 通过：形状 {returns10.shape}；末端误差带 [{band_lower[-1]:.3f}, {band_upper[-1]:.3f}] 整体 > 0")

## 🏔 挑战（无答案）：ε 线性衰减 vs 恒定

恒定 ε 有个内置缺陷：训练后期，探索还在白白撞墙。经典修补是**衰减**——ε 从 1.0 线性衰减到 0.1：开局多探索（攒样本），后期少探索（收割）。

问题：**哪个样本效率高？高多少？优势在第几回合显现？**

要求（把本笔记本的习惯用上）：

- 多 seed（≥ 10），误差带口径（第 7 节同款）——两条带分不开就别下结论；
- 报告"达到 greedy 评估 ≥ 0.6 所需回合数"的中位数对比；
- 提示在下格注释里；答案没有——这正是科研训练的味道。

In [ ]:
# 🏔 挑战 starter（无参考答案）——把下面的骨架补完，或推倒重写
#
# def train_decay(seed=42, episodes=EPISODES, eps_start=1.0, eps_end=0.1, alpha=0.1):
#     rng = np.random.default_rng(seed)
#     ...
#     for e in range(episodes):
#         epsilon = eps_start + (eps_end - eps_start) * e / (episodes - 1)   # 线性衰减
#         ...   # 其余同 train()，只是 epsilon 逐回合变化
#
# 对照组：train(seed=..., epsilon=0.1)（恒定）。评价指标建议：
#   1) 各回合的 greedy 评估回报什么时候第一次 >= 0.6（对 10 个 seed 取中位数）；
#   2) 训练期回报曲线 + 第 7 节的误差带画法，两组各一条带；
#   3) 若两条带大部分重叠——结论就只能是"差异不显著"，这本身也是合格的实验结论。
#
# 我的结论（写在这里）：
#

pass  # 本格不设 assert、无答案——跑通你的实验就是交付

## 8 · 回顾：你刚刚做了什么

- **复现**：`train(seed, alpha, epsilon, episodes)`——同 seed 同曲线，随机性全部收口到一个 rng；
- **α 敏感性**：大步长学得快、稳态更抖；确定性环境里经典"早期抖动"让位于"稳态抖"；
- **ε 敏感性**：训练期探索再烂（ε=1.0 也在内），greedy 评估照样最优——off-policy 的解耦；
- **误差带**：均值 ± 1.96·std/√n——从"画一条线"升级到"报告一个区间"；
- 顺带：乐观初始化怎么救活全零初始化的平局偏置。

下一本 **NB5（L8 · TD 线性近似）**：状态不再是一格一 ID，而是特征向量——Q 表装不下的世界从这里开始。带上今天的多 seed 习惯，那边的方差问题只会更凶。